#### Tool Binding

In [1]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.messages import ToolMessage
from langchain_core.messages import SystemMessage
from dotenv import load_dotenv
import requests

In [2]:
load_dotenv()

True

In [3]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

In [4]:
print(multiply.invoke({'a':2, 'b':5}))

10


In [5]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [6]:
# tool binding

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.5
)

llm_with_tools = llm.bind_tools([multiply])

#### Tool Calling

In [7]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm functioning properly, thank you for asking. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 234, 'total_tokens': 252, 'completion_time': 0.027105516, 'completion_tokens_details': None, 'prompt_time': 0.016906666, 'prompt_tokens_details': None, 'queue_time': 0.046745884, 'total_time': 0.044012182}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c8c1b-154e-7110-be02-603859e0468f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 18, 'total_tokens': 252})

In [8]:
query = HumanMessage('can you multiply 3 with 10')

In [ ]:
messages = [
    SystemMessage(
        content="You are a helpful assistant. Use tools only when necessary. After receiving the tool result, respond with the final answer and do not call tools again."
    )
]

In [10]:
messages.append(query)

In [11]:
messages

[SystemMessage(content='You are a helpful assistant. Use tools only when necessary. After receiving the tool result, respond with the final answer and do not call tools again.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={})]

In [12]:
result = llm_with_tools.invoke(messages)

In [13]:
messages.append(result)

In [14]:
messages

[SystemMessage(content='You are a helpful assistant. Use tools only when necessary. After receiving the tool result, respond with the final answer and do not call tools again.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q1tw36jze', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 268, 'total_tokens': 297, 'completion_time': 0.049425925, 'completion_tokens_details': None, 'prompt_time': 0.018395027, 'prompt_tokens_details': None, 'queue_time': 0.046279493, 'total_time': 0.067820952}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c8c1b-16d7-72a3-8f27-bd349e548c94-

In [15]:
result.tool_calls[0]['args']

{'a': 3, 'b': 10}

In [16]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'q1tw36jze',
 'type': 'tool_call'}

In [17]:
tool_result = multiply.invoke(result.tool_calls[0])

In [18]:
messages.append(tool_result)
messages

[SystemMessage(content='You are a helpful assistant. Use tools only when necessary. After receiving the tool result, respond with the final answer and do not call tools again.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q1tw36jze', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 268, 'total_tokens': 297, 'completion_time': 0.049425925, 'completion_tokens_details': None, 'prompt_time': 0.018395027, 'prompt_tokens_details': None, 'queue_time': 0.046279493, 'total_time': 0.067820952}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c8c1b-16d7-72a3-8f27-bd349e548c94-

In [19]:
llm_with_tools.invoke(messages).content

'The final answer is 30.'

In [20]:
max_steps = 5
step = 0

while step < max_steps:
    step += 1
    result = llm_with_tools.invoke(messages)
    messages.append(result)

    if result.tool_calls:
        tool_call = result.tool_calls[0]
        tool_output = multiply.invoke(tool_call["args"])

        messages.append(
            ToolMessage(
                content=str(tool_output),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )
    else:
        print("Final Answer:", result.content)
        break

if step == max_steps:
    print("Stopped due to max steps")

Final Answer: The final answer is $\boxed{30}$.
